# CI/CD 第4周：进阶实战 — 可复用工作流、自定义 Action、通知与安全

> **学习目标**：掌握 Reusable Workflows、Custom Actions、通知集成、安全最佳实践，搭建生产级 CI/CD 流水线

---

## 本周概览

前三周你学会了：
1. 基本的 Workflow 编写（第1周）
2. Matrix、缓存、Artifacts 等进阶配置（第2周）
3. Docker 构建与 K8s 部署（第3周）

本周要解决的问题：
- 多个项目有相同的 lint/test 配置，**怎么复用**？
- 标准 Action 不够用，**怎么自己写 Action**？
- CI 完成后**怎么自动通知团队**？
- CI/CD 中有**哪些安全风险**？怎么防范？
- **怎么省钱**？怎么优化 Action 成本？

前 6 天学新内容，最后 2 天做综合项目。

---

## Day 22：Reusable Workflows（可复用工作流）

### 为什么需要 Reusable Workflows？

假设你的团队有 5 个 Python 微服务，每个都要 lint + test + build。如果每个仓库各自写一份配置，改一次 lint 规则（比如从 `ruff` 换成 `flake8`）要在 5 个仓库重复改 5 次。

**Reusable Workflow** = 一份配置，多处调用。

### 定义 Reusable Workflow

和普通 Workflow 的区别：
- 用 `on: workflow_call` 代替 `on: push`
- 通过 `inputs` 和 `secrets` 接收参数

```yaml
# .github/workflows/python-ci.yml (在共享仓库中)
name: Python CI Reusable

on:
  workflow_call:
    inputs:
      python-version:
        required: true
        type: string
        default: '3.12'
      run-mypy:
        required: false
        type: boolean
        default: false
    secrets:
      CODECOV_TOKEN:
        required: false

jobs:
  lint:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: \${{ inputs.python-version }}
      - run: pip install ruff
      - run: ruff check .

  test:
    needs: lint
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: \${{ inputs.python-version }}
      - uses: actions/cache@v4
        with:
          path: ~/.cache/pip
          key: pip-\${{ hashFiles('**/requirements.txt') }}
      - run: pip install -r requirements.txt
      - run: pytest
      - if: \${{ inputs.run-mypy }}
        run: mypy .
```

### 调用 Reusable Workflow

```yaml
# 在另一个仓库中调用
name: CI
on: push

jobs:
  call-python-ci:
    uses: my-org/shared-workflows/.github/workflows/python-ci.yml@main
    with:
      python-version: '3.11'
      run-mypy: true
    secrets:
      CODECOV_TOKEN: \${{ secrets.CODECOV_TOKEN }}
```

调用时用 `uses` 指定：
```
uses: <org>/<repo>/.github/workflows/<file>@<ref>
```

**好处**：
- 一处修改，所有调用方自动生效
- 通过参数和行为差异
- 团队可以统一维护一套 CI 标准

### DRY 原则

Reusable Workflow 贯彻了软件工程的 **DRY（Don't Repeat Yourself）** 原则：

```
❌ 坏做法：5 个仓库复制 5 份相同的 CI 配置
✅ 好做法：1 个共享仓库定义，5 个仓库分别引用
```

In [ ]:
# 模拟 Reusable Workflow 的调用链

import time

print("=" * 60)
print("🔄 Reusable Workflow 调用演示")
print("=" * 60)

print("\n📂 共享仓库: shared-workflows")
print("   文件: .github/workflows/python-ci.yml")
print("   版本: main")

repos = [
    ("auth-service", "3.12", True, True),
    ("payment-service", "3.11", False, True),
    ("notification-service", "3.12", True, True),
    ("api-gateway", "3.11", True, True),
    ("user-service", "3.12", False, False),
]

print("\n📋 调用方:")
for repo, py_version, mypy, codecov in repos:
    print(f"\n  📁 {repo}")
    print(f"     python-version: {py_version}")
    print(f"     run-mypy: {mypy}")
    print(f"     codecov: {codecov}")
    time.sleep(0.2)
    print(f"     ✅ CI 执行完成")

print("\n" + "-" * 40)
print("💡 如果 lint 规则变了:")
print("   只需要改 shared-workflows 一个文件")
print("   5 个仓库下次调用时自动使用新规则")

### 练习

把之前的 test + lint workflow 抽成 reusable workflow，在另一个仓库中引用它。

---

## Day 23：Custom Actions（自定义 Action）

### Action 的三种类型

| 类型 | 复杂度 | 适用场景 |
|------|--------|----------|
| **Composite Action** | ⭐ | 组合多个 Step（最轻量，推荐） |
| **JavaScript Action** | ⭐⭐⭐ | 需要 Node.js 逻辑处理 |
| **Docker Action** | ⭐⭐ | 需要在容器中运行 |

### Composite Action（最常用）

用一个 `action.yml` 文件组合多个 Step：

```yaml
# .github/actions/python-setup-test/action.yml
name: 'Python Setup and Test'
description: 'Setup Python, install deps, run tests'
inputs:
  python-version:
    description: 'Python version'
    required: true
    default: '3.12'
  working-directory:
    description: 'Working directory'
    required: false
    default: '.'
outputs:
  test-result:
    description: 'Test result (passed/failed)'
    value: \${{ steps.run-tests.outcome }}

runs:
  using: 'composite'
  steps:
    - uses: actions/checkout@v4

    - uses: actions/setup-python@v5
      with:
        python-version: \${{ inputs.python-version }}

    - uses: actions/cache@v4
      with:
        path: ~/.cache/pip
        key: pip-\${{ hashFiles('**/requirements.txt') }}

    - run: pip install -r requirements.txt
      shell: bash
      working-directory: \${{ inputs.working-directory }}

    - id: run-tests
      run: pytest || echo "tests failed"
      shell: bash
      working-directory: \${{ inputs.working-directory }}
```

**在 Workflow 中使用**：

```yaml
steps:
  - uses: ./.github/actions/python-setup-test
    with:
      python-version: '3.11'
      working-directory: ./backend
```

In [ ]:
# 模拟 Composite Action 的执行

import time

print("=" * 60)
print("🔧 Composite Action 执行模拟")
print("=" * 60)

action_def = '''
name: Python Setup and Test
description: 安装 Python 环境并运行测试
inputs:
  python-version:
    default: "3.12"
  working-directory:
    default: "."
runs:
  using: composite
  steps:
    - actions/checkout@v4
    - actions/setup-python@v5
    - actions/cache@v4
    - run: pip install -r requirements.txt
    - run: pytest
'''

print("\n📄 Action 定义:")
for line in action_def.strip().split('\n'):
    print(f"  {line}")

print("\n\n▶ 执行 Action (input: python-version=3.11)")
steps = ["checkout", "setup-python (3.11)", "cache (key: pip-xxx)", "pip install", "pytest"]
for step in steps:
    print(f"   🔄 {step}...")
    time.sleep(0.15)
    print(f"   ✅ {step} 完成")

print("\n✅ Composite Action 执行成功！")
print("   输出: test-result = 'success'")
print("   用时: 45.2s")

### 练习

写一个 Composite Action：接受 Python 版本和包名作为 input，自动 setup → install → pytest → 输出结果。

---

## Day 24：通知集成

### 为什么需要通知？

CI/CD 是自动运行的——你不会一直盯着 Actions 页面看。当 CI 失败或部署完成时，**自动通知**让团队即时得知结果。

### Slack 通知

```yaml
- name: Notify Slack on success
  uses: slackapi/slack-github-action@v1
  if: success()
  with:
    channel-id: 'C0123456789'
    slack-message: |
      ✅ Deployment to production succeeded!
      *Version:* \${{ github.ref_name }}
      *Commit:* \${{ github.sha }}
  env:
    SLACK_BOT_TOKEN: \${{ secrets.SLACK_BOT_TOKEN }}
```

### 飞书/企业微信 Webhook

```yaml
- name: Notify on failure
  if: failure()
  run: |
    curl -X POST -H "Content-Type: application/json" \
      -d '{"msgtype": "text", "text": {"content": "❌ CI failed in ${{ github.repository }}"}}' \
      \${{ secrets.FEISHU_WEBHOOK_URL }}
```

### GitHub Issue（失败自动创建 Issue）

```yaml
- name: Create Issue on failure
  if: failure()
  uses: actions/github-script@v7
  with:
    script: |
      await github.rest.issues.create({
        owner: context.repo.owner,
        repo: context.repo.repo,
        title: \`CI Failed: \${{ github.workflow }} #\${{ github.run_number }}\`,
        body: \`❌ Workflow failed on commit \${{ github.sha }}

  See details: \${{ github.server_url }}/\${{ github.repository }}/actions/runs/\${{ github.run_id }}
        \`
      });
```

In [ ]:
# 模拟通知集成

import time

print("=" * 60)
print("🔔 CI/CD 通知集成演示")
print("=" * 60)

# 场景: CI 完成
notifications = [
    ("CI 成功 ✅", "Slack", ["#dev-team", "@channel CI passed!"]),
    ("CI 失败 ❌", "飞书 Webhook", ["POST webhook", "❌ 服务构建失败"]),
    ("CI 失败 ❌", "GitHub Issue", ["创建 Issue #142", "CI Failed: Build #87"]),
]

for scenario, channel, details in notifications:
    print(f"\n📋 场景: {scenario}")
    print(f"   通道: {channel}")
    for detail in details:
        print(f"   → {detail}")
    time.sleep(0.3)
    print(f"   ✅ 通知已发送")

print("\n" + "-" * 40)
print("💡 推荐的通知策略:")
print("  - 成功: 发送到团队 Slack 频道（摘要）")
print("  - 失败: 发送到个人 + 创建 Issue（可追踪）")
print("  - 部署完成: 发送到相关频道（含版本信息）")

### 练习

在 Workflow 的末尾加上通知 Step：成功发 Slack 通知（或飞书 Webhook），失败创建 GitHub Issue。

---

## Day 25：CI/CD 安全

### 1. OIDC 认证（不用长期 Token）

传统做法是在 Secrets 中存 AWS/GCP 的长期密钥。**不安全**，密钥泄露风险大。

**OIDC（OpenID Connect）** 让 GitHub Actions 和工作负载身份联邦：
- 每次运行时，GitHub 向云服务商（AWS/GCP/Azure）颁发**短时令牌**
- 无需在 Secrets 中存储任何云厂商的密钥

```yaml
# AWS 部署示例（使用 OIDC）
permissions:
  id-token: write  # OIDC 需要
  contents: read

steps:
  - name: Configure AWS credentials
    uses: aws-actions/configure-aws-credentials@v4
    with:
      role-to-assume: arn:aws:iam::123456:role/GitHubActionsRole
      aws-region: us-east-1

  - run: aws ecs update-service ...
```

### 2. 最小权限原则

默认的 `permissions` 太宽了：

```yaml
# ❌ 不好：write-all 权限
permissions: write-all

# ✅ 好：只给需要的权限
permissions:
  contents: read
  packages: write
  issues: write  # 只需要创建 Issue 时才给
```

**在 Job 级别也可以设置**，进一步缩小权限范围：

```yaml
jobs:
  deploy:
    permissions:
      contents: read
      packages: write
    runs-on: ubuntu-latest
    steps:
      - ...
```

In [ ]:
# 模拟权限设置对比

print("=" * 60)
print("🔒 CI/CD 安全权限对比")
print("=" * 60)

print("\n❌ 不安全的权限配置:")
print('''
permissions: write-all
  ├── contents: write  ← 不必要的写权限
  ├── packages: write
  ├── issues: write
  ├── pull-requests: write
  ├── actions: write
  └── ...
''')

print("✅ 最小权限配置:")
print('''
permissions:
  contents: read     # 只需要读代码
  packages: write    # 需要推送镜像
''')

print("✅ Job 级权限（更精细）:")
print('''
jobs:
  test:
    permissions:
      contents: read     # test 只需要读
    ...

  deploy:
    permissions:
      contents: read
      packages: write    # deploy 需要写包
      id-token: write    # OIDC 认证
    ...
''')

print("\n⚠️ 安全提醒:")
print("  1. 用 OIDC 替代长期密钥")
print("  2. 只给必要的权限")
print("  3. 第三方 Action 要审核代码")
print("  4. 不要在日志中打印 Secret")
print("  5. 定期轮换 Secrets")

### 3. 第三方 Action 风险

使用第三方 Action 时要注意：
- **Pin 到 commit SHA**而非 tag（避免 tag 指向恶意版本）
- 审查 Action 的源代码（特别是安装脚本）
- 尽量使用官方 Action（`actions/*`）

```yaml
# ❌ 不安全的引用方式（tag 可能被改动）
- uses: some-user/action@v1

# ✅ 安全的引用方式（pin 到 commit SHA）
- uses: some-user/action@a1b2c3d4e5f6...
```

### 练习

审查之前所有 workflow 的 `permissions` 设置，把默认的 `write-all` 收紧为仅需要的权限。

---

## Day 26：成本优化

### GitHub Actions 的计费

| 套餐 | 免费额度（每月） |
|------|----------------|
| GitHub Free | 2000 分钟/月 |
| GitHub Team | 3000 分钟/月 |
| GitHub Enterprise | 50000 分钟/月 |

**省钱策略**：

### 1. 跳过不必要的运行

```yaml
on:
  push:
    paths-ignore:
      - 'README.md'
      - 'docs/**'
      - '*.md'
      - '.gitignore'
```

文档变更不需要跑 CI！`paths-ignore` 可以节省大量 Action 时长。

### 2. 并发控制

```yaml
# 同一个分支的旧运行自动取消
concurrency:
  group: \${{ github.workflow }}-\${{ github.ref }}
  cancel-in-progress: true
```

如果你在 dev 分支上快速连续提交 5 次，前 4 次 CI 其实不用跑完——反正会被第 5 次覆盖。

### 3. 选择合适的 Runner

- `ubuntu-latest` **最便宜**（免费额度最多）
- `macos-latest` 贵约 10 倍（只有 iOS/macOS 构建才需要）
- 只在不必要时用 `windows-latest`

### 4. 缓存

`actions/cache` 和 Docker layer 缓存不仅加速，也省钱——Runner 运行时间缩短，计费时间减少。

In [ ]:
# 模拟成本优化效果

import time

print("=" * 60)
print("💰 成本优化效果演示")
print("=" * 60)

# 优化前
before = {
    "每月运行次数": 200,
    "平均运行时间": "4分30秒",
    "总分钟数": 900,
    "paths-ignore": "未配置",
    "并发控制": "未配置",
    "缓存": "未配置"
}

after = {
    "每月运行次数": 200,
    "平均运行时间": "1分45秒",
    "总分钟数": 350,
    "paths-ignore": "已配置（节省 ~30% 触发）",
    "并发控制": "已配置",
    "缓存": "已配置（节省 ~60% 时间）"
}

print("\n📊 优化前后对比:")
print(f"{'项目':<20} {'优化前':<20} {'优化后':<20}")
print("-" * 60)
print(f"{'每月运行次数':<20} {str(before['每月运行次数']):<20} {str(after['每月运行次数']):<20}")
print(f"{'平均运行时间':<20} {before['平均运行时间']:<20} {after['平均运行时间']:<20}")
print(f"{'总分钟数/月':<20} {str(before['总分钟数']):<20} {str(after['总分钟数']):<20}")
print(f"{'paths-ignore':<20} {before['paths-ignore']:<20} {after['paths-ignore']:<20}")
print(f"{'并发控制':<20} {before['并发控制']:<20} {after['并发控制']:<20}")
print(f"{'缓存':<20} {before['缓存']:<20} {after['缓存']:<20}")

savings = round((before['总分钟数'] - after['总分钟数']) / before['总分钟数'] * 100)
print(f"\n🎯 总节省: {savings}%（{before['总分钟数'] - after['总分钟数']} 分钟/月）")

### 练习

给你最常用的仓库设置 `concurrency` 规则，并配置 `paths-ignore` 跳过 README 等非代码变更。

---

## 🎯 第4周总结（前6天）

| 主题 | 核心要点 |
|------|----------|
| **Reusable Workflows** | `on: workflow_call` + `uses:` 跨仓库复用 |
| **Composite Action** | `action.yml` 组合多个 Step，团队内共享 |
| **通知集成** | Slack / 飞书 / GitHub Issue，成功和失败不同处理 |
| **安全** | OIDC、最小权限、pin 第三方 Action、Secret 保护 |
| **成本优化** | `paths-ignore`、`concurrency`、缓存、选择合适的 Runner |

这些技能让你从一个"能写 CI"的人变成了一个"能设计 CI 体系"的人。

---

## 第27-28天：综合项目 — 完整的 CI/CD 流水线

为一个真实项目（可以是你的 Python 学习项目）搭建完整的 CI/CD 流水线。

### 项目全景图

```
GitHub 仓库 CI/CD 全景：

1. PR 阶段
   ├── lint (ruff + mypy)
   ├── test (matrix: 3 Python × 2 OS, 带缓存)
   └── security scan (pip-audit / safety)

2. Merge 到 main 后
   ├── build Docker image (layer 缓存)
   ├── push to GHCR (latest + commit-sha)
   ├── deploy to dev K8s (自动)
   └── 通知 Slack "dev 已更新"

3. 发版 Tag (v*)
   ├── build Docker image
   ├── push (latest + tag + commit-sha)
   ├── deploy to staging K8s (需要审批)
   └── 跑一遍 E2E 测试

4. 每日定时
   ├── 运行全部测试 (schedule cron)
   └── 若失败创建 Issue
```

In [ ]:
# 模拟完整 CI/CD 全景的执行

import time

print("=" * 70)
print("🏗️ 完整 CI/CD 全景模拟")
print("=" * 70)

# Phase 1: PR
print("\n" + "=" * 50)
print("📌 Phase 1: PR 阶段 [#133 feat: add user profile]")
print("=" * 50)
time.sleep(0.3)
print("  ✅ lint (ruff + mypy): 通过")
print("  ✅ test matrix (3.10, 3.11, 3.12 × ubuntu, macos): 全部通过")
print("  ✅ security scan (pip-audit): 0 已知漏洞")
print("  ✅ Status Check: 全部通过, Merge 按钮变绿")
print("  📊 耗时: 3m 42s")

# Phase 2: Merge to main
print("\n" + "=" * 50)
print("📌 Phase 2: Merge 到 main")
print("=" * 50)
time.sleep(0.4)
print("  ✅ GitHub Actions 自动触发")
print("  ✅ 构建 Docker 镜像 (layer 缓存命中, 15s)")
print("  ✅ 推送至 GHCR")
print("     latest → ghcr.io/my-org/my-app@sha256:abc123")
print("     a1b2c3d → ghcr.io/my-org/my-app:a1b2c3d")
print("     main   → ghcr.io/my-org/my-app:main")
print("  ✅ 自动部署到 dev 环境")
print("     触发滚动更新, 3/3 就绪")
print("  ✅ 通知: Slack #dev 频道 → "dev 已更新 (a1b2c3d)"")

# Phase 3: Tag release
print("\n" + "=" * 50)
print("📌 Phase 3: 发版 v2.1.0")
print("=" * 50)
time.sleep(0.4)
print("  ✅ 构建 & 推送镜像: v2.1.0, latest, a1b2c3d")
print("  ⏳ 等待审批: @tech-lead 审批中...")
time.sleep(0.5)
print("  ✅ @tech-lead 已批准")
print("  ✅ 部署到 staging 集群")
print("     kubectl rollout status: ✅ 完成")
print("  ✅ E2E 测试: 全部通过 (24 tests, 0 failures)")
print("  ✅ 通知: Slack #release → "v2.1.0 已部署到 staging"")

# Phase 4: Daily
print("\n" + "=" * 50)
print("📌 Phase 4: 每日定时任务")
print("=" * 50)
time.sleep(0.3)
print("  ⏰ cron: 0 2 * * * (每天 UTC 2:00)")
print("  ✅ 全部测试通过")
print("  ✅ 依赖安全检查通过")
print("  ✅ 报告已归档到 Artifacts")

print("\n" + "=" * 70)
print("🎉 生产级 CI/CD 流水线运行正常！")
print("=" * 70)

### 完整项目 Workflow 配置

此综合项目包含了前 26 天学到的全部知识点：

```yaml
name: Full CI/CD Pipeline
on:
  push:
    branches: [main]
    tags: ['v*']
    paths-ignore:
      - 'README.md'
      - 'docs/**'
  pull_request:
    branches: [main]
  schedule:
    - cron: '0 2 * * *'
  workflow_dispatch:
    inputs:
      environment:
        description: '部署环境'
        type: choice
        options: [dev, staging]

concurrency:
  group: \${{ github.workflow }}-\${{ github.ref }}
  cancel-in-progress: true

jobs:
  # 1. Lint (PR / push 触发)
  lint:
    if: github.event_name != 'schedule'
    uses: ./.github/workflows/lint-reusable.yml

  # 2. Test Matrix (PR / push / schedule 触发)
  test:
    strategy:
      fail-fast: false
      matrix:
        python-version: ['3.10', '3.11', '3.12']
        os: [ubuntu-latest, macos-latest]
    runs-on: \${{ matrix.os }}
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: \${{ matrix.python-version }}
      - uses: actions/cache@v4
        with:
          path: ~/.cache/pip
          key: \${{ runner.os }}-pip-\${{ hashFiles('**/requirements.txt') }}
      - run: pip install -r requirements.txt
      - run: pytest --cov=xml
      - uses: actions/upload-artifact@v4
        with:
          name: coverage-\${{ matrix.os }}-\${{ matrix.python-version }}
          path: coverage.xml

  # 3. Security Scan
  security:
    if: github.event_name == 'pull_request'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: pip install pip-audit
      - run: pip-audit

  # 4. Build & Push (仅 main push / tag push)
  build-and-push:
    if: github.event_name == 'push' && github.ref_name == 'main'
    needs: [lint, test]
    runs-on: ubuntu-latest
    permissions:
      contents: read
      packages: write
    steps:
      - uses: actions/checkout@v4
      - uses: docker/login-action@v3
        with:
          registry: ghcr.io
          username: \${{ github.actor }}
          password: \${{ secrets.GITHUB_TOKEN }}
      - uses: docker/metadata-action@v5
        id: meta
        with:
          images: ghcr.io/\${{ github.repository }}
          tags: |
            type=sha,format=short
            type=ref,event=branch
            type=semver,pattern=v{{version}}
            type=raw,value=latest
      - uses: docker/build-push-action@v6
        with:
          push: true
          tags: \${{ steps.meta.outputs.tags }}
          cache-from: type=gha
          cache-to: type=gha,mode=max

  # 5. Deploy to Dev (main push)
  deploy-dev:
    if: github.event_name == 'push' && github.ref_name == 'main'
    needs: [build-and-push]
    environment: dev
    runs-on: ubuntu-latest
    steps:
      - run: echo "kubectl apply -f k8s/dev/"

  # 6. Deploy to Staging (tag push, 需要审批)
  deploy-staging:
    if: startsWith(github.ref, 'refs/tags/v')
    needs: [build-and-push]
    environment: staging
    runs-on: ubuntu-latest
    steps:
      - run: echo "kubectl apply -f k8s/staging/"

  # 7. Notification (failures → Issue)
  notify-failure:
    if: failure() && github.event_name != 'schedule'
    needs: [lint, test, build-and-push]
    runs-on: ubuntu-latest
    permissions:
      issues: write
    steps:
      - uses: actions/github-script@v7
        with:
          script: |
            github.rest.issues.create({
              owner: context.repo.owner,
              repo: context.repo.repo,
              title: \`CI Failed: \${{ github.run_id }}\`,
              body: \`Workflow failed in \${{ github.repository }}\`
            })
```

---

## 🏆 学习路线里程碑

### 你已掌握的技能

```
Week 1: ✅ 能写基本的 push/PR 触发 workflow，理解 Job/Step/Runner 关系
Week 2: ✅ 能配置 matrix build、缓存、artifacts、secrets，设置 Branch Protection
Week 3: ✅ 能在 CI 中构建 Docker 镜像并部署到 K8s，管理多环境
Week 4: ✅ 能编写 reusable workflow、custom action，搭建生产级 CI/CD 流水线
```

### 下一步学习方向

| 方向 | 资源 |
|------|------|
| GitHub Actions 官方文档 | [docs.github.com/en/actions](https://docs.github.com/en/actions) |
| Awesome Actions | [github.com/sdras/awesome-actions](https://github.com/sdras/awesome-actions) |
| GitHub Skills 互动教程 | [github.com/skills](https://github.com/skills/) |
| 阅读大项目配置 | 查看知名开源项目的 `.github/workflows/` 目录 |
| 持续交付理念 | 《持续交付》(Jez Humble) |

### 最后的建议

1. **从小开始，逐步迭代**：先写一个最小 workflow，再逐步加功能
2. **本地测试**：用 `act` 工具在本地运行 GitHub Actions（`nektos/act`）
3. **阅读他人的配置**：GitHub 上大项目的 workflow 是最好的学习材料
4. **保持简单**：不要过度设计，满足需求即可
5. **安全第一**：永远检查权限和 Secret 的使用

---

## 🧪 最终练习：动手搭建你的 CI/CD 流水线

现在，是时候把学到的知识付诸实践了。

### 步骤指南

1. **选一个项目**：可以是你的 Python 学习项目、Web 应用或任何 GitHub 仓库
2. **从 PR 检查开始**：写 lint + test workflow
3. **添加缓存**：给依赖安装添加缓存加速
4. **添加 Matrix**：如果支持多版本，加上 Matrix 配置
5. **添加 Docker 构建**：创建 Dockerfile，在 CI 中构建镜像
6. **推送到 GHCR**：配置镜像推送
7. **添加部署**：如果有 K8s 集群，配置部署步骤
8. **添加通知**：CI 成功/失败通知到团队聊天工具
9. **检查安全**：收紧 permissions，检查 Secret 使用
10. **优化成本**：配置 paths-ignore 和 concurrency

### 自我检查清单

- [ ] 能写出不依赖图形界面的完整 workflow
- [ ] 理解 `needs`、`matrix`、`if` 等关键语法
- [ ] 能解释 Runner 和 Job 的关系
- [ ] 知道如何排查失败的 workflow（看日志、加 debug 输出）
- [ ] 理解 Secrets 和 Variables 的区别
- [ ] 知道 OIDC 是什么，为什么比长期密钥更安全
- [ ] 能设计适合团队的多环境部署策略

---

> 🎉 **恭喜完成 CI/CD 学习路线！你现在已经具备了在生产环境中设计和维护 CI/CD 流水线的能力。**